In [1]:
import os
import sys
import time
import requests
import rasterio
import geopandas as gpd
from osgeo import gdal
import math
import numpy as np

origin = '/data/Aldhani/eoagritwin/'
sys.path.append('/home/potzschf/repos/')

import pandas as pd
from datetime import datetime
from helperToolz.helper import getFilelist
from helperToolz.helpsters import dirfinder, force_order_Colors_for_VRT, force_to_vrt, path_safe, reduce_forceTSA_output_to_validmonths

year = 2023
force_path = f"{origin}force/output/GERMANY/{year}/"
vrt_path = f"{origin}fields/Auxiliary/vrt/Gohar_paper/{year}"

colorL = ['BLUE', 'GREEN', 'RED', 'BROADNIR']


In [17]:
bioL=[False, [1,2,3]] 
C_HEIGHTL=['lai','fix']
T_HEIGHTL=['high','low']
LAND_CL=['fix','th']
combis = [[bi, chei, thei, lanc] for bi in bioL for chei in C_HEIGHTL for thei in T_HEIGHTL for lanc in LAND_CL]
bioL, C_HEIGHTL, T_HEIGHTL, LAND_CL = map(list, zip(*combis))
print(len(bioL))

16


In [13]:
path_to_dem = '/data/Aldhani/eoagritwin/et/Auxiliary/DEM/Force_Tiles/DEM/'

In [14]:
pathDEML =[False, path_to_dem] 
C_HEIGHTL=['lai','fix']
T_HEIGHTL=['high','low']
LAND_CL=['fix','th']
combis = [[dm, chei, thei, lanc] for dm in pathDEML for chei in C_HEIGHTL for thei in T_HEIGHTL for lanc in LAND_CL]
pathDEML, C_HEIGHTL, T_HEIGHTL, LAND_CL = map(list, zip(*combis))

In [18]:
len(pathDEML)

16

In [8]:
from helperToolz.helpsters import convertVRTpathsTOrelative


tiles = dirfinder(force_path)

for tile in tiles:
     t_files = getFilelist(f"{force_path}{tile}/", '.tif', deep=True)
     ordered_files = force_order_Colors_for_VRT(t_files, colorL, [f'MONTH-{d:02d}' for d in range(3,9,1)])
     ordered_files = [ordfile[0] for ordfile in ordered_files]
     
     vrt_out = path_safe(f'{vrt_path}/{tile}.vrt')
     vrt = gdal.BuildVRT(vrt_out, ordered_files, separate = True)
     vrt = None

     # make paths in vrts relative
     convertVRTpathsTOrelative(vrt_out)     
     for idz, bname in enumerate(np.repeat(colorL,int(len(ordered_files) / len(colorL))).tolist()):
          if idz < 6:
               b_name = f"{bname}_{idz + 1}"
          elif 5 < idz < 12:
               b_name = f"{bname}_{idz - 6 + 1}"
          elif 11 < idz < 18:
               b_name = f"{bname}_{idz - 12 + 1}"
          else:
               b_name = f"{bname}_{idz - 18 + 1}"
          vrt = gdal.Open(vrt_out, gdal.GA_Update)  # VRT must be writable
          band = vrt.GetRasterBand(idz + 1)
          band.SetDescription(b_name)
          vrt = None


In [9]:
path_to_polygon = "/data/Aldhani/eoagritwin/fields/01_IACS/1_Polygons/LSA/GSA-DE_LSA-2023.geoparquet"
df = gpd.read_parquet(path_to_polygon)

In [13]:
df['EC_trans_n']

0         DGL reseeding as a replacement for approved DG...
1                                         Winter soft wheat
2                                         Winter soft wheat
3                                           Mowing pastures
4                                           Mowing pastures
                                ...                        
889321                                      Mowing pastures
889322                                      Mowing pastures
889323                                      Mowing pastures
889324                               Pome fruit e.g. apples
889325                               Pome fruit e.g. apples
Name: EC_trans_n, Length: 889326, dtype: object

In [ ]:
def force_to_vrt(list_of_forcefiles, ordered_forcetiles, vrt_out_path, pyramids=False, bandnames=False):
    '''list_of_forcefiles: e.g. output from reduce_force_to_validmonths
        ordered_forcetiles: e.g output from getCOLORSinOrderFORCELIST (single=False)
        vrt_out_path: path where .vrt files will be created (there will be more than one to account for all the bands)
        pyramids: if set to True, pyramids will be created (might be very very large!!)'''
    
    # tiles = list(set([file.split('output/')[-1].split('/')[1].split('/')[0] for file in list_of_forcefiles]))
    force_folder_name = getFORCExyRangeName(get_forcetiles_range(list_of_forcefiles))
    if not vrt_out_path.endswith('/'):
        vrt_out_path = vrt_out_path + '/'
    outDir = f'{vrt_out_path}{force_folder_name}/'
    if not os.path.exists(outDir):
        os.makedirs(outDir)
        print(outDir)
        vrts = []
        for i in range(len(ordered_forcetiles)):
            vrt_name = f'{outDir}{force_folder_name}_{str(i)}.vrt'
            vrt = gdal.BuildVRT(vrt_name, ordered_forcetiles[i], separate = False)
            vrt = None

            # make paths in vrts relative
            convertVRTpathsTOrelative(vrt_name)
            vrts.append(vrt_name)

        # set optionally bandnames    
        if bandnames:
            for idz, bname in enumerate(np.repeat(bandnames,int(len(ordered_forcetiles) / len(bandnames))).tolist()):  
                print(f'{outDir}{force_folder_name}_{str(idz)}.vrt')
                vrt = gdal.Open(f'{outDir}{force_folder_name}_{str(idz)}.vrt', gdal.GA_Update)  # VRT must be writable
                band = vrt.GetRasterBand(1)
                band.SetDescription(bname)
                vrt = None
        print('single vrts created')
        
        nums = [int(vrt.split('_')[-1].split('.')[0]) for vrt in vrts]
        vrts_sorted = sortListwithOtherlist(nums, vrts)[-1]
        print('paths in vrts made relative')
        
        vrt = gdal.BuildVRT(f'{outDir}{force_folder_name}_Cube.vrt', vrts_sorted, separate = True)
        vrt = None
        if bandnames:
            # set vrt band names
            vrt = gdal.Open(f'{outDir}{force_folder_name}_Cube.vrt', gdal.GA_Update)  # VRT must be writable
            for idz, bname in enumerate(np.repeat(bandnames,int(len(ordered_forcetiles) / len(bandnames))).tolist()): 
                band = vrt.GetRasterBand(1+idz)
                band.SetDescription(bname)
            vrt = None
        # convertVRTpathsTOrelative(f'{outDir}{force_folder_name}_Cube.vrt')
        print('overlord vrt created')
        if pyramids:
            # build pyramids
            vrtPyramids(f'{outDir}{force_folder_name}_Cube.vrt')
            print('VRT created with pyramids')
    else:
        print('Vrt might already exist - please check!!')


In [ ]:
vrt_out = path_safe(f'{origin}fields/Auxiliary/vrt/{state}/{year}/')
reduced_files = reduce_forceTSA_output_to_validmonths(f'{origin}force/output/{state}/{year}/', 3, 8)
ordered_files = force_order_Colors_for_VRT(reduced_files, colorList, [f'MONTH-{d:02d}' for d in range(3,9,1)])

In [ ]:
reduced_files

In [ ]:
# zip masked chips
unmasked_folder = '/data/Aldhani/eoagritwin/fields/04_Predictions/GERMANY/FromScratch_IACS_dilate_True_BorderEdgeCutted_RGB_NDVI_exclude_True_with_overlap_47/2023/chips_folder/unmasked_chips/'
mfiles = getFilelist(unmasked_folder, '.tif')
print(len(mfiles))
n = 32
chunk_size = math.ceil(len(mfiles) / n)
chunks = [mfiles[i:i + chunk_size] for i in range(0, len(mfiles), chunk_size)]

In [ ]:
def zip_chunk(args):
    idx, chunk, masked_folder = args

    zip_name = os.path.join(masked_folder, f"unmasked_{idx}.zip")

    with zipfile.ZipFile(
        zip_name,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as zf:
        for tif_path in chunk:
            zf.write(tif_path, arcname=os.path.basename(tif_path))

print('zipping masked chips')
with ProcessPoolExecutor(max_workers=200) as executor:
    executor.map(
        zip_chunk,
        [(idx, chunk, unmasked_folder) for idx, chunk in enumerate(chunks)]
    )



In [ ]:

##########################################################################

mask = (arr_field != 0) & (~np.isnan(arr_field)) & (~np.isnan(arr_b1))
flat_field = arr_field[mask].astype(int)
flat_b1 = arr_b1[mask]

# Create DataFrame
df = pd.DataFrame({'FieldID': flat_field, 'B1': flat_b1})

# Compute median per field
median_df = df.groupby('FieldID', as_index=False)['B1'].median()
median_df.rename(columns={'B1': 'B1_median'}, inplace=True)
median_df.to_csv(path_safe(f"{origin}output/uncertainty/{aoi}_mean_b1_score_per_field_ext{ext}_bound{bound}.csv"), index=False)

# Create a lookup map
median_map = dict(zip(median_df['FieldID'], median_df['B1_median']))

# Fill raster
arr_field_fill = np.vectorize(median_map.get)(arr_field)
# get_value = lambda x: median_map.get(x, np.nan)
# arr_field_fill = np.vectorize(get_value, otypes=[float])(arr_field)

In [ ]:
arr_expo = np.where(arr_field_fill == None, np.nan, arr_field_fill)


In [ ]:
arr_export = arr_export.astype(np.float32)
arr_export[236:-2766,1416:] = arr_expo

npTOdisk(arr_export, path_fields,
         f"{origin}output/uncertainty/{aoi}_Fields_MEDIAN_B1_SCORE_ext{ext}_bound{bound}.tif",
         noData=0, d_type=gdal.GDT_Float32)


In [ ]:
print(ext_b1)
print(ext_fields)

In [ ]:
print((ext_b1[3] - ext_fields[3])/10)
print((ext_b1[1] - ext_fields[1])/10)

print((ext_fields[2] - ext_b1[2])/10)
print((ext_b1[0] - ext_fields[0])/10)

In [ ]:
tt = arr_field[236:-2766,1416:]
print(tt.shape)
qq = arr_b1[:,:-69]
print(qq.shape)

In [ ]:
# create a mask for clean fields for uncertainty maps
pathi = f"{origin}output/uncertainty/{aoi}_Fields_MEDIAN_B1_SCORE_ext{ext}_bound{bound}.tif"
pathi2 = f"{origin}output/uncertainty/{aoi}_Borders_MEDIAN_B2_SCORE_ext{ext}_bound{bound}.tif"
ds = gdal.Open(pathi)
ds2 = gdal.Open(pathi2)
arr = ds.GetRasterBand(1).ReadAsArray()
arr2 = ds2.GetRasterBand(1).ReadAsArray()
arr_export = zeros = np.zeros_like(arr)
arr = arr[236:-2766,1416:]
arr2 = arr2[:,:-69]
# arr_np = np.where(np.logical_or(arr == 0, np.isnan(arr)), 1, 0)
mask1 = (arr == 0) | np.isnan(arr)
mask2 = (arr2 == 0) | np.isnan(arr2)
arr_np = np.where(mask1 & mask2, 1, 0)

arr_export = arr_export.astype(np.float32)
arr_export[236:-2766,1416:] = arr_np
npTOdisk(arr_export, pathi,
         f"{origin}output/uncertainty/MASK_ext{ext}_bound{bound}.tif",
         noData=0, d_type=gdal.GDT_Byte)

In [ ]:
ext_bounds = getExtentRas(pathi2)
ext_fields = getExtentRas(pathi)

print(ext_bounds)
print(ext_fields)

In [ ]:
print((ext_bounds['Xmin'] - ext_fields['Xmin']) / 10)
print((ext_bounds['Xmax'] - ext_fields['Xmax']) / 10)
print((ext_bounds['Ymin'] - ext_fields['Ymin']) / 10)
print((ext_bounds['Ymax'] - ext_fields['Ymax']) / 10)


In [ ]:
arr[236:-2766,1416:].shape

In [ ]:
arr2[:,:-69].shape

In [ ]:
# cut em all into smaller pieces for easier map creation
field_path = f"{origin}output/uncertainty/{aoi}_Fields_MEDIAN_B1_SCORE_ext{ext}_bound{bound}.tif"
border_path = f"{origin}output/uncertainty/{aoi}_Borders_MEDIAN_B2_SCORE_ext{ext}_bound{bound}.tif"
mask_path = f"{origin}output/uncertainty/MASK_ext{ext}_bound{bound}.tif"

evaps = 

In [ ]:
subset_mask_to_prediction_extent

In [ ]:
from FieldWaterUseTools.FuncBox.Misc import getFilelist, path_safe


year = 2023
master = f"{origin}fields/04_Predictions/GERMANY/FromScratch_IACS_dilate_True_BorderEdgeCutted_RGB_NDVI_exclude_True_with_overlap_47/{year}/"
chips_folder = f"{master}chips_folder/unmasked_chips/"
masked_folder = path_safe(f"{master}chips_folder/masked_chips/")
files = getFilelist(chips_folder, '.tif')
ds = gdal.Open(f"{master}vrt/Masked_THUENEN_CTM_2023.tif")

In [ ]:
conti = []

for file in files:
    # load masked stack
    chip = ds.GetRasterBand(1).ReadAsArray(
        xoff=int(file.split('X_')[-1].split('_')[0]),
        yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
        win_xsize=236,  
        win_ysize=236 
    )

    if np.nansum(chip) > 0:

        with rasterio.open(file) as src:
            bounds = src.bounds  # left, bottom, right, top
            geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            crs = src.crs

            conti.append({
                "filename": os.path.basename(file),
                "geometry": geom,
                "crs_used": str(crs)
            })

            chip2 = ds.GetRasterBand(2).ReadAsArray(
                xoff=int(file.split('X_')[-1].split('_')[0]),
                yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
                win_xsize=236,  
                win_ysize=236 
            )

            chip3 = ds.GetRasterBand(3).ReadAsArray(
                xoff=int(file.split('X_')[-1].split('_')[0]),
                yoff=int(file.split('X_')[-1].split('_')[2].split('.')[0]),
                win_xsize=236,  
                win_ysize=236 
            )

            stack = np.stack([chip, chip2, chip3], axis=0)

            with rasterio.open(
                f"{masked_folder}chips_masked256{file.split('_unmasked256')[-1]}",
                "w",
                driver="GTiff",
                height=stack.shape[1],
                width=stack.shape[2],
                count=stack.shape[0],
                dtype=stack.dtype,
                crs=crs,          # z.B. von einem Referenz-Datensatz: ref_ds.crs
                transform=src.transform  # z.B. ref_ds.transform
            ) as dst:
                dst.write(stack)

In [ ]:
gdf = gpd.GeoDataFrame(conti, geometry="geometry", crs=conti[0]["crs_used"])
gdf.to_file(f"{master}{year}_grid_tiles.gpkg", driver="GPKG", layer="tiles")

In [ ]:
from FieldWaterUseTools.FuncBox.FieldFuncis import predicted_chips_to_vrt

predicted_chips_to_vrt(f"{master}chips_folder/", 'masked_chips', 256, 20, path_safe(f"{master}vrt/"), pyramids=True)

In [ ]:
# zip em
mfiles = getFilelist(masked_folder, '.tif')
print(len(mfiles))
n = 8
chunk_size = math.ceil(len(mfiles) / n)
chunks = [mfiles[i:i + chunk_size] for i in range(0, len(mfiles), chunk_size)]

In [ ]:
len(chunks[0])

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def zip_chunk(args):
    idx, chunk, masked_folder = args

    zip_name = os.path.join(masked_folder, f"masked_{idx}.zip")

    with zipfile.ZipFile(
        zip_name,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as zf:
        for tif_path in chunk:
            zf.write(tif_path, arcname=os.path.basename(tif_path))



with ProcessPoolExecutor(max_workers=30) as executor:
    executor.map(
        zip_chunk,
        [(idx, chunk, masked_folder) for idx, chunk in enumerate(chunks)]
    )